In [5]:
import pandas as pd
import numpy as np

# 1. Load Data (Watch out for the double .csv if your file has it!)
df = pd.read_csv('customer_churn_nn.csv')

print("--- First 5 Rows ---")
display(df.head())

# Drop the ID column if it exists (change 'customerID' if yours is named differently)
# Based on the kernel state, 'customerID' is present and should be dropped.
if 'customerID' in df.columns:
    df = df.drop('customerID', axis=1)

print("\n--- Target Class Balance ---")
# ASSUMPTION: Your target column is named 'Churn'. Change it if needed!
# To find the correct column name, uncomment the line below to inspect your DataFrame's columns:
# print(df.columns)
target_col = 'churn' # <--- This needs to be updated to the actual target column name in your data
print(df[target_col].value_counts())

--- First 5 Rows ---


,customer_id,region,plan_type,contract_type,payment_method,tenure_months,monthly_charges_inr,avg_login_days_per_month,support_tickets_last_90_days,payment_delay_days,data_usage_gb,satisfaction_score,last_complaint_days_ago,discount_percent,autopay_enabled,referral_count,churn
0,CUST0001,South,Standard,Month-to-month,Debit Card,30,687.40,13,0,0,87.97,8.0,67,0,0,0,0
1,CUST0002,West,Premium,Month-to-month,Wallet,15,1029.74,22,3,1,82.17,5.7,69,0,0,0,0
2,CUST0003,Central,Standard,Month-to-month,Credit Card,72,732.07,13,0,11,89.39,6.4,63,10,0,0,0
3,CUST0004,West,Premium,Month-to-month,Credit Card,22,959.51,19,2,3,139.73,7.2,130,5,0,0,0
4,CUST0005,North,Premium,Month-to-month,Net Banking,11,890.20,18,2,6,156.43,5.8,0,5,1,2,0



--- Target Class Balance ---
churn
0    1969
1      31
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Convert 'Yes'/'No' target into 1 and 0
df[target_col] = df[target_col].map({'Yes': 1, 'No': 0, 1: 1, 0: 0})

# Separate features (X) and target (y)
X = df.drop(target_col, axis=1)
y = df[target_col]

# Convert all text columns into numbers automatically (One-Hot Encoding)
X = pd.get_dummies(X, drop_first=True)

# Train-Test Split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the numbers (Very important for Neural Networks!)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Data successfully preprocessed! Shape of training data:", X_train.shape)

Data successfully preprocessed! Shape of training data: (1600, 2023)


In [7]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Build the Neural Network
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.2), # Prevents overfitting
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid') # Sigmoid is used for Binary Classification (Yes/No)
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
print("--- Training Neural Network ---")
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.2)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


--- Training Neural Network ---
Epoch 1/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.9828 - loss: 0.1173 - val_accuracy: 0.9781 - val_loss: 0.1221
Epoch 2/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9867 - loss: 0.0685 - val_accuracy: 0.9781 - val_loss: 0.1195
Epoch 3/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9867 - loss: 0.0443 - val_accuracy: 0.9781 - val_loss: 0.1209
Epoch 4/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9883 - loss: 0.0301 - val_accuracy: 0.9781 - val_loss: 0.1218
Epoch 5/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9922 - loss: 0.0196 - val_accuracy: 0.9781 - val_loss: 0.1247
Epoch 6/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9937 - loss: 0.0127 - val_accuracy: 0.9781 - val_loss: 0.1251
Epoch 7/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9937 - loss: 0.0093 - val_accuracy: 0.9781 - val_loss: 0.1300
Epoch 8/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9984 - loss: 0.006

In [8]:
# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Final Test Accuracy: {accuracy*100:.2f}%")

# 1. Save Model Evaluation as CSV
eval_df = pd.DataFrame({'Metric': ['Loss', 'Accuracy'], 'Value': [loss, accuracy]})
eval_df.to_csv('model_evaluation.csv', index=False)

# 2. Save Sample Predictions as TXT
predictions = model.predict(X_test[:5])
predicted_classes = (predictions > 0.5).astype(int)

with open('sample_predictions.txt', 'w') as f:
    f.write("--- Customer Churn Sample Predictions ---\n")
    for i in range(5):
        f.write(f"True Label: {y_test.iloc[i]} | Predicted: {predicted_classes[i][0]}\n")

print("Files generated! Check the sidebar to download model_evaluation.csv and sample_predictions.txt.")

Final Test Accuracy: 98.25%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
Files generated! Check the sidebar to download model_evaluation.csv and sample_predictions.txt.
